# 📘 Krones Vision AI Challenge — Train + Infer Notebook
### ✅ ConvNeXt-Large ROI Classifier (Train, ONNX Export, Test Submission)

This notebook follows the **official starter training structure** and runs end-to-end on Kaggle:

✅ Step-by-step structured workflow  
✅ ROI extraction from COCO annotations (category 22)  
✅ ConvNeXt-Large fine-tuning (timm, ImageNet pretrained)  
✅ Training + validation F1 with early stopping  
✅ Dual-checkpoint ensemble inference (best + last epoch)  
✅ **ONNX export** (required for the Evaluation Notebook)  
✅ `submission.csv` generation  

---

## ⚠️ Important Notes on Kaggle Environment Requirements

### 🔌 Internet Access & Pretrained Weights

This notebook downloads **ImageNet pretrained weights** via `timm` and installs `onnxscript`.

- Enable **Internet** in Notebook Settings  
- **Phone verification** is required by Kaggle for internet + GPU access  

### ⚡ Hardware Accelerators

Select **GPU T4 x1** (or P100). Training is tuned for a **≤ 12 hour** wall-clock budget.

### 🔒 Competition Requirement

You may modify any part of this notebook — **except**:

🔥 The **ONNX export must remain functional**, as it is required for the **Evaluation Notebook**.

---

## 🎯 Approach Summary

Binary ROI classification (reusable vs non-reusable bottles): crop bottle region, fine-tune ConvNeXt-Large with EMA + MixUp + focal loss, calibrate threshold on a stratified 8% validation split, ensemble **best** and **last** checkpoints at test time with 4-view TTA, export best weights to ONNX.


# ✅ Step 1 — Check Your Hardware

Verify GPU/CPU availability before training.

In [ ]:
!nvidia-smi

In [ ]:
!cat /proc/cpuinfo | head -n 20

# ✅ Step 2 — Install Dependencies, Import Libraries & Configure Paths

Install `timm` and `onnxscript`, import libraries, and set Kaggle competition paths.

In [ ]:
!pip install -q timm onnxscript

# =========================================================
# Imports
# =========================================================
import json
import os
import random
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torchvision.transforms.v2 as T
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from timm.data import Mixup
from timm.utils import ModelEmaV2
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

NOTEBOOK_START = time.perf_counter()

# =========================================================
# Dataset Paths — Kaggle mounts competition data here
# =========================================================
DATASET_DIR = "/kaggle/input/competitions/1st-krones-vision-ai-challenge"
TRAIN_IMAGE_DIR = os.path.join(DATASET_DIR, "train_images")
TEST_IMAGE_DIR = os.path.join(DATASET_DIR, "test_images")
TRAIN_CSV = os.path.join(DATASET_DIR, "train.csv")
TRAIN_JSON = os.path.join(DATASET_DIR, "train_annotations.json")
TEST_JSON = os.path.join(DATASET_DIR, "test_annotations_roi_only.json")
SAMPLE_SUBMISSION = os.path.join(DATASET_DIR, "sample_submission.csv")
BOTTLETYPES_CSV = os.path.join(DATASET_DIR, "bottletypes.csv")

ARTIFACTS_DIR = "training_artifacts"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# =========================================================
# Hyperparameters (edit here — total run <= 12 h on T4)
# =========================================================
CONFIG = {
    "model_name": "convnext_large",
    "imgsz": 320,
    "batch_size": 4,
    "grad_accum": 4,
    "max_epochs": 12,
    "patience": 4,
    "val_fraction": 0.08,
    "lr": 3e-5,
    "weight_decay": 0.08,
    "warmup_epochs": 2,
    "use_ema": True,
    "use_mixup": True,
    "use_dual_checkpoint_ensemble": True,
    "tta_n": 4,
    "infer_batch_size": 16,
    "num_workers": 2,
    "max_runtime_hours": 12,
    "reserve_infer_hours": 1.5,
}

MAX_RUNTIME_SEC = CONFIG["max_runtime_hours"] * 3600
RESERVE_INFER_SEC = CONFIG["reserve_infer_hours"] * 3600

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)
print("Dataset directory exists:", os.path.exists(DATASET_DIR))
print("Train images:", os.path.exists(TRAIN_IMAGE_DIR))
print("Test images:", os.path.exists(TEST_IMAGE_DIR))
print("Train CSV:", os.path.exists(TRAIN_CSV))
print("Annotations JSON:", os.path.exists(TRAIN_JSON))


# ✅ Step 3 — Load ROI (Region of Interest) From COCO JSON

Category **22** marks the bottle ROI in training annotations.  
Test ROI boxes come from `test_annotations_roi_only.json`.

In [ ]:
def load_roi_map(annotation_file, category_id=22, require_category=True):
    """
    Loads COCO annotations and extracts ROI bounding boxes.
    Returns: { filename : [x, y, w, h] }
    """
    with open(annotation_file, "r") as f:
        data = json.load(f)

    img_lookup = {img["id"]: img["file_name"] for img in data["images"]}
    roi_map = {}

    for ann in data["annotations"]:
        if require_category and ann.get("category_id") != category_id:
            continue
        fname = img_lookup[ann["image_id"]]
        roi_map[fname] = ann["bbox"]

    print(f"✅ Loaded {len(roi_map)} ROI regions from {os.path.basename(annotation_file)}.")
    return roi_map


def load_all_rois():
    roi_map = load_roi_map(TRAIN_JSON, category_id=22, require_category=True)
    if os.path.exists(TEST_JSON):
        test_rois = load_roi_map(TEST_JSON, category_id=22, require_category=False)
        roi_map.update(test_rois)
    return roi_map


def load_bottletypes():
    if not os.path.exists(BOTTLETYPES_CSV):
        print("Warning: bottletypes.csv not found.", flush=True)
        return {}
    df = pd.read_csv(BOTTLETYPES_CSV)
    return dict(zip(df["image_id"], df["bottle_type"]))


def elapsed_hours():
    return (time.perf_counter() - NOTEBOOK_START) / 3600.0


def remaining_train_budget_sec():
    return MAX_RUNTIME_SEC - RESERVE_INFER_SEC - (time.perf_counter() - NOTEBOOK_START)


print("✅ ROI loader functions defined successfully")


# ✅ Step 4 — Build PyTorch Datasets (with ROI Cropping)

Train/validation datasets apply augmentations; test dataset returns PIL images for TTA.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def build_train_transform(imgsz):
    oversize = int(imgsz * 1.15)
    return T.Compose([
        T.Resize((oversize, oversize)),
        T.RandomCrop(imgsz),
        T.RandomHorizontalFlip(0.5),
        T.RandomVerticalFlip(0.5),
        T.RandomRotation(15),
        T.ColorJitter(0.4, 0.4, 0.4, 0.08),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        T.RandomApply([T.RandomPerspective(0.2, p=1.0)], p=0.3),
        T.RandomApply([T.RandomAutocontrast(p=1.0)], p=0.3),
        T.RandomApply([T.RandomAdjustSharpness(2.0, p=1.0)], p=0.3),
        T.RandomErasing(p=0.3, scale=(0.02, 0.2)),
    ])


def build_val_transform(imgsz):
    oversize = int(imgsz * 1.08)
    return T.Compose([
        T.Resize((oversize, oversize)),
        T.CenterCrop(imgsz),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class RoiDataset(Dataset):
    """Train/val dataset with ROI crop + transforms."""

    def __init__(self, df, roi_map, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.roi_map = roi_map
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row["image_id"]
        target = torch.tensor(row["target"], dtype=torch.long)
        img = Image.open(os.path.join(self.image_dir, fname)).convert("RGB")
        if fname in self.roi_map:
            x, y, w, h = self.roi_map[fname]
            img = img.crop((x, y, x + w, y + h))
        if self.transform:
            img = self.transform(img)
        return img, target


class TestRoiDataset(Dataset):
    """Test dataset — returns PIL image + index for TTA inference."""

    def __init__(self, df, roi_map, image_dir):
        self.df = df.reset_index(drop=True)
        self.roi_map = roi_map
        self.image_dir = image_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row["image_id"]
        path = os.path.join(self.image_dir, fname)
        if not os.path.exists(path):
            return None, idx
        img = Image.open(path).convert("RGB")
        if fname in self.roi_map:
            x, y, w, h = self.roi_map[fname]
            img = img.crop((x, y, x + w, y + h))
        return img, idx


def collate_test(batch):
    imgs, indices = [], []
    for img, idx in batch:
        if img is not None:
            imgs.append(img)
            indices.append(idx)
    return imgs, indices


print("✅ Dataset classes defined successfully")


# ✅ Step 5 — Create the Model (ConvNeXt-Large via timm)

ImageNet-pretrained backbone with a 2-class head (reusable vs non-reusable).

In [ ]:
def create_model(model_name=None):
    """Create timm classifier with ImageNet pretrained weights."""
    name = model_name or CONFIG["model_name"]
    model = timm.create_model(
        name,
        pretrained=True,
        num_classes=2,
        drop_rate=0.3,
        drop_path_rate=0.2,
    )
    return model


class SoftTargetFocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.log_softmax = nn.LogSoftmax(dim=-1)

    def forward(self, inputs, targets):
        log_probs = self.log_softmax(inputs)
        probs = torch.exp(log_probs)
        return (-torch.sum(targets * (1 - probs) ** self.gamma * log_probs, dim=-1)).mean()


def find_best_threshold(y_true, y_probs, steps=199):
    y_true = np.asarray(y_true)
    y_probs = np.asarray(y_probs)
    best_f1, best_t = 0.0, 0.5
    for t in np.linspace(0.01, 0.99, steps):
        score = f1_score(y_true, (y_probs >= t).astype(int), zero_division=0)
        if score > best_f1:
            best_f1, best_t = score, t
    return best_t, best_f1


class OnnxExportWrapper(nn.Module):
    """Export reusable-class logit as single output (starter-compatible shape)."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        logits = self.model(x)
        return logits[:, 1:2]


print("✅ Model creation functions defined successfully")


# ✅ Step 6 — Training Loop

One epoch: forward, focal loss, AMP, gradient accumulation, EMA update.

In [ ]:
def train_one_epoch(model, loader, optimizer, epoch, criterion, scaler, ema, mixup_fn, accum):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    use_amp = device.type == "cuda"

    pbar = tqdm(loader, desc=f"Training Epoch {epoch}")

    for step, (imgs, targets) in enumerate(pbar, start=1):
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if mixup_fn is not None:
            imgs, targets = mixup_fn(imgs, targets)

        with torch.amp.autocast("cuda", enabled=use_amp):
            loss = criterion(model(imgs), targets) / accum

        scaler.scale(loss).backward()

        if step % accum == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(model)

        total_loss += loss.item() * accum
        pbar.set_postfix({"Loss": f"{loss.item() * accum:.4f}"})

    avg_loss = total_loss / len(loader)
    print(f"✅ Epoch {epoch} — Avg Loss: {avg_loss:.4f}")
    return avg_loss


print("✅ Training loop function defined successfully")


# ✅ Step 7 — Validation Loop

Computes validation loss and searches for the F1-optimal probability threshold.

In [ ]:
@torch.inference_mode()
def validate(eval_model, loader, val_criterion):
    eval_model.eval()
    y_true, y_probs, val_loss = [], [], 0.0
    use_amp = device.type == "cuda"

    pbar = tqdm(loader, desc="Validating")
    for imgs, targets in pbar:
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            out = eval_model(imgs)
            val_loss += val_criterion(out, targets).item()
        y_probs.extend(torch.softmax(out, 1)[:, 1].cpu().numpy())
        y_true.extend(targets.cpu().numpy())

    val_loss /= max(len(loader), 1)
    thresh, f1 = find_best_threshold(y_true, y_probs)
    return {"f1": f1, "threshold": thresh, "val_loss": val_loss}


print("✅ Validation function defined successfully")


# ✅ Step 8 — Load Data and Initialize DataLoaders

Load ROI map, CSV labels, stratified train/val split, and build DataLoaders.

In [ ]:
# Create output folder
os.makedirs(os.path.join(os.getcwd(), ARTIFACTS_DIR), exist_ok=True)
run_name = "run_" + datetime.now().strftime("%Y%m%d_%H%M")
run_dir = os.path.join(ARTIFACTS_DIR, run_name)
os.makedirs(run_dir, exist_ok=True)

best_ckpt_path = os.path.join(run_dir, "best_model.pth")
last_ckpt_path = os.path.join(run_dir, "latest_model.pth")
onnx_path = os.path.join(run_dir, "model.onnx")

IMGSZ = CONFIG["imgsz"]
train_tf = build_train_transform(IMGSZ)
val_tf = build_val_transform(IMGSZ)

roi_map = load_all_rois()
bottletypes = load_bottletypes()

df = pd.read_csv(TRAIN_CSV)
sample_df = pd.read_csv(SAMPLE_SUBMISSION)

# Full training set (set LIMIT for quick debugging)
LIMIT = None
if LIMIT is not None:
    df = df.sample(LIMIT, random_state=SEED).reset_index(drop=True)

train_df, val_df = train_test_split(
    df,
    test_size=CONFIG["val_fraction"],
    stratify=df["target"],
    random_state=SEED,
)

train_ds = RoiDataset(train_df, roi_map, TRAIN_IMAGE_DIR, transform=train_tf)
val_ds = RoiDataset(val_df, roi_map, TRAIN_IMAGE_DIR, transform=val_tf)

loader_kw = {
    "num_workers": CONFIG["num_workers"],
    "pin_memory": device.type == "cuda",
}
if CONFIG["num_workers"] > 0:
    loader_kw["persistent_workers"] = True
    loader_kw["prefetch_factor"] = 2

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    **loader_kw,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["batch_size"] * 2,
    shuffle=False,
    **loader_kw,
)

print(f"✅ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(sample_df)}")
print(f"✅ Artifacts dir: {run_dir}")


# ✅ Step 9 — Dataset Review & Visualization

Visualize ROI crops and augmentations from the training stream.

In [ ]:
def show_samples(dataset, num_per_class=4):
    class_samples = {0: [], 1: []}

    for idx in range(len(dataset)):
        img, label = dataset[idx]
        fname = dataset.df.iloc[idx]["image_id"]
        label = int(label.item())
        if len(class_samples[label]) < num_per_class:
            class_samples[label].append((img, fname))
        if len(class_samples[0]) >= num_per_class and len(class_samples[1]) >= num_per_class:
            break

    fig, ax = plt.subplots(2, num_per_class, figsize=(num_per_class * 3.5, 7))
    mean = np.array(IMAGENET_MEAN)
    std = np.array(IMAGENET_STD)

    for cls in [0, 1]:
        for i, (img, fname) in enumerate(class_samples[cls]):
            img_np = img.permute(1, 2, 0).numpy()
            img_np = std * img_np + mean
            img_np = np.clip(img_np, 0, 1)
            ax[cls, i].imshow(img_np)
            ax[cls, i].set_title(f"{fname[:28]}...\nclass {cls}", fontsize=8)
            ax[cls, i].axis("off")

    plt.tight_layout()
    plt.show()


show_samples(train_ds)


# ✅ Step 10 — Run Full Training Pipeline

Trains up to `max_epochs` with early stopping and saves:

- `best_model.pth` (best validation F1)  
- `latest_model.pth` (last epoch)  

Stops automatically when the 12 h budget (minus inference reserve) is reached.

In [ ]:
# RUNTIME BUDGET GUIDE (total <= 12 h on T4):
# max_epochs=12, tta_n=4, dual ckpt -> ~9-11 h train + ~1 h infer (default)
# max_epochs=8,  tta_n=4, dual ckpt -> ~6-8 h train + ~1 h infer (safer)
# use_dual_checkpoint_ensemble=False -> ~half infer time

TRAIN_START = time.perf_counter()
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)

model = create_model().to(device)
ema = ModelEmaV2(model, decay=0.9998, device=device) if CONFIG["use_ema"] else None
mixup_fn = (
    Mixup(mixup_alpha=0.8, cutmix_alpha=1.0, prob=0.9, switch_prob=0.5,
          mode="batch", label_smoothing=0.1, num_classes=2)
    if CONFIG["use_mixup"] else None
)
criterion = SoftTargetFocalLoss(gamma=2.0)
val_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = SequentialLR(
    optimizer,
    [
        LinearLR(optimizer, start_factor=0.01, total_iters=CONFIG["warmup_epochs"]),
        CosineAnnealingLR(
            optimizer,
            T_max=max(CONFIG["max_epochs"] - CONFIG["warmup_epochs"], 1),
            eta_min=5e-8,
        ),
    ],
    milestones=[CONFIG["warmup_epochs"]],
)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
accum = CONFIG["grad_accum"]

best_f1 = 0.0
best_thresh = 0.5
best_epoch = 0
last_epoch = 0
patience_counter = 0

for epoch in range(1, CONFIG["max_epochs"] + 1):
    if remaining_train_budget_sec() <= 0:
        print(f"⏱️ Stopping: training budget exhausted before epoch {epoch}", flush=True)
        break

    t0 = time.perf_counter()
    train_one_epoch(model, train_loader, optimizer, epoch, criterion, scaler, ema, mixup_fn, accum)
    scheduler.step()

    eval_model = ema.module if ema is not None else model
    metrics = validate(eval_model, val_loader, val_criterion)
    last_epoch = epoch

    torch.save(eval_model.state_dict(), last_ckpt_path)

    ep_min = (time.perf_counter() - t0) / 60.0
    print(
        f"Epoch {epoch}/{CONFIG['max_epochs']} | val_loss={metrics['val_loss']:.4f} "
        f"| F1={metrics['f1']:.4f} @ thresh={metrics['threshold']:.4f} "
        f"| {ep_min:.1f} min/ep | elapsed={elapsed_hours():.2f} h",
        flush=True,
    )

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_thresh = metrics["threshold"]
        best_epoch = epoch
        patience_counter = 0
        torch.save(eval_model.state_dict(), best_ckpt_path)
        print("✅ New Best Model Saved!")
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["patience"]:
            print(f"Early stop @ epoch {epoch} (patience={CONFIG['patience']})", flush=True)
            break

if not os.path.exists(best_ckpt_path):
    raise RuntimeError("No checkpoint saved — verify dataset paths and GPU.")

threshold = best_thresh
train_hours = (time.perf_counter() - TRAIN_START) / 3600.0
print(f"\n✅ Training done in {train_hours:.2f} h | best F1={best_f1:.4f} | threshold={threshold:.4f}")
print(f"   best (ep {best_epoch}): {best_ckpt_path}")
print(f"   latest (ep {last_epoch}): {last_ckpt_path}")


# ✅ Step 11 — Test Inference & Submission

Run TTA inference with best (+ latest) checkpoints, average probabilities, write `submission.csv`.

In [ ]:
def tta_views(img, n):
    w, h = img.size
    c = int(min(w, h) * 0.85)
    left, top = (w - c) // 2, (h - c) // 2
    zoom = img.crop((left, top, left + c, top + c)).resize((w, h), Image.BICUBIC)
    views = [
        img,
        img.transpose(Image.FLIP_LEFT_RIGHT),
        img.transpose(Image.FLIP_TOP_BOTTOM),
        img.transpose(Image.FLIP_LEFT_RIGHT).transpose(Image.FLIP_TOP_BOTTOM),
        img.rotate(5, Image.BICUBIC),
        img.rotate(-5, Image.BICUBIC),
        img.rotate(10, Image.BICUBIC),
        img.rotate(-10, Image.BICUBIC),
        img.rotate(90, Image.BICUBIC),
        img.rotate(180, Image.BICUBIC),
        img.rotate(270, Image.BICUBIC),
        zoom,
    ]
    return views[:n]


INFER_START = time.perf_counter()
tta_n = CONFIG["tta_n"]
use_amp = device.type == "cuda"

test_ds = TestRoiDataset(sample_df, roi_map, TEST_IMAGE_DIR)
test_loader = DataLoader(
    test_ds,
    batch_size=CONFIG["infer_batch_size"],
    shuffle=False,
    collate_fn=collate_test,
    num_workers=CONFIG["num_workers"],
    pin_memory=device.type == "cuda",
)

infer_ckpts = [("best", best_ckpt_path)]
if CONFIG["use_dual_checkpoint_ensemble"] and os.path.exists(last_ckpt_path) and last_epoch != best_epoch:
    infer_ckpts.append(("latest", last_ckpt_path))
elif CONFIG["use_dual_checkpoint_ensemble"] and last_epoch == best_epoch:
    print("Dual ensemble skipped: latest epoch equals best epoch.", flush=True)

n_ckpts = len(infer_ckpts)
print(f"Inference checkpoints ({n_ckpts}): {[x[0] for x in infer_ckpts]}", flush=True)

n_test = len(sample_df)
prob_sum = np.zeros(n_test, dtype=np.float64)


@torch.inference_mode()
def run_checkpoint_infer(ckpt_path, label):
    infer_model = create_model()
    infer_model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    infer_model.to(device).eval()

    for imgs, indices in tqdm(test_loader, desc=f"Infer {label}", leave=False):
        batch_tensors, batch_map = [], []
        for j, img in enumerate(imgs):
            for view in tta_views(img, tta_n):
                batch_tensors.append(val_tf(view))
                batch_map.append(j)
        if not batch_tensors:
            continue
        x = torch.stack(batch_tensors).to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            batch_probs = torch.softmax(infer_model(x), dim=1)[:, 1].cpu().numpy()
        per_img = [[] for _ in range(len(imgs))]
        for prob, j in zip(batch_probs, batch_map):
            per_img[j].append(float(prob))
        for j, idx in enumerate(indices):
            prob_sum[idx] += float(np.mean(per_img[j]))

    del infer_model
    if device.type == "cuda":
        torch.cuda.empty_cache()


for label, path in tqdm(infer_ckpts, desc="Checkpoints", leave=True):
    print(f"Running {label}: {path}", flush=True)
    run_checkpoint_infer(path, label)

probs = prob_sum / n_ckpts
preds = (probs >= threshold).astype(int)

submission = sample_df[["image_id"]].copy()
submission["target"] = preds
submission.to_csv(SUBMISSION_PATH, index=False)

infer_min = (time.perf_counter() - INFER_START) / 60.0
print(f"✅ Inference done in {infer_min:.1f} min | positive rate: {100*preds.mean():.2f}%")
print(f"✅ Saved submission: {SUBMISSION_PATH}")

if bottletypes:
    print(f"\n{'bottle_type':<40} | {'n':>6} | {'pos%':>8} | {'mean_p':>8}", flush=True)
    diag = pd.DataFrame({
        "bottle_type": [bottletypes.get(i, "unknown") for i in sample_df["image_id"]],
        "pred": preds,
        "prob": probs,
    })
    for btype, grp in diag.groupby("bottle_type", sort=False):
        print(f"{str(btype):<40} | {len(grp):6d} | {100*grp['pred'].mean():7.2f}% | {grp['prob'].mean():8.4f}", flush=True)


# ✅ Step 12 — Export Model to ONNX Format (Required!)

Export the **best** checkpoint to ONNX for the Official Evaluation Notebook.

⚠️ Do not remove this step.

✅ If you see `Saved ONNX model to ...` the export was successful.

In [ ]:
def export_to_onnx(model, imgsz, save_path):
    """Export PyTorch model to ONNX (single reusable-class logit output)."""
    model.eval()
    wrapper = OnnxExportWrapper(model).to(device)
    dummy = torch.randn(1, 3, imgsz, imgsz, device=device)

    torch.onnx.export(
        wrapper,
        dummy,
        save_path,
        input_names=["input"],
        output_names=["logits"],
        opset_version=18,
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    )
    print(f"✅ Saved ONNX model to {save_path}")


export_model = create_model()
export_model.load_state_dict(torch.load(best_ckpt_path, map_location=device, weights_only=True))
export_model.to(device)
export_to_onnx(export_model, IMGSZ, onnx_path)


# ✅ Step 13 — Runtime Summary

Total wall-clock time and configuration recap.

In [ ]:
elapsed_h = elapsed_hours()
print(f"Training time:  {train_hours:.2f} h")
print(f"Inference time: {infer_min:.1f} min")
print(f"Total elapsed:  {elapsed_h:.2f} h")

if device.type == "cuda":
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated(device) / 1024**3:.2f} GB")

print(f"Model: {CONFIG['model_name']} | imgsz={IMGSZ} | checkpoints={n_ckpts} | TTA={tta_n}")
print(f"Best val F1: {best_f1:.4f} | Threshold: {threshold:.4f}")
print(f"Artifacts: {run_dir}")

if elapsed_h > CONFIG["max_runtime_hours"]:
    print(f"⚠️ Exceeded {CONFIG['max_runtime_hours']} h — reduce max_epochs or use convnext_base.")
elif elapsed_h > CONFIG["max_runtime_hours"] * 0.9:
    print("Note: >90% of budget used — consider fewer epochs for Efficiency score.")
